In [1]:
import os
from sqlalchemy import create_engine
from jinja2 import Environment, FileSystemLoader
import pandas as pd
import geopandas as gpd
import shapely
import shapely.ops
import shapely.wkt

from src.data_loading import (
    get_matching_shapes_feeds,
    get_schedule_feed_entries,
    get_schedule_rt_stop_times,
    get_schedule_rt_headways
)
from src.metric_calculation import calculate_travel_time_variability, calculate_expected_wait, calculate_on_time_performance

## Configuration

In [2]:
TARGET_DATES = [
    "2026-06-15","2026-06-16","2026-06-17","2026-06-18","2026-06-19"
]
TARGET_ROUTE_SHORT_NAME = os.environ.get("TARGET_ROUTE_SHORT_NAME", "ECR")
TARGET_SCHEDULE_FEED = os.environ.get("TARGET_SCHEDULE_FEED", "Bay Area 511 SamTrans Schedule")
CONSTANT_MACROS = {
    "TARGET_DATES": TARGET_DATES,
    "TARGET_ROUTE_SHORT_NAME": TARGET_ROUTE_SHORT_NAME,
    "TARGET_SCHEDULE_FEED": TARGET_SCHEDULE_FEED
}
CONSTANT_MACROS

{'TARGET_DATES': ['2026-06-15',
  '2026-06-16',
  '2026-06-17',
  '2026-06-18',
  '2026-06-19'],
 'TARGET_ROUTE_SHORT_NAME': 'ECR',
 'TARGET_SCHEDULE_FEED': 'Bay Area 511 SamTrans Schedule'}

In [3]:
# min count of trips per service period to be considered reliable
MIN_TRIPS = 10

# early/late definition for OTP
EARLY_SECONDS = 60
LATE_SECONDS = 5 * 60

## Data/Calculation

In [4]:
# gcp boilerplate
CALITP_BQ_MAX_BYTES = os.environ.get("CALITP_BQ_MAX_BYTES", 5_000_000_000)
CALITP_BQ_LOCATION = os.environ.get("CALITP_BQ_LOCATION", "us-west2")
engine = create_engine(
    f"bigquery://cal-itp-data-infra/?maximum_bytes_billed={CALITP_BQ_MAX_BYTES}",  # noqa: E231
    location=CALITP_BQ_LOCATION
)
env = Environment(loader=FileSystemLoader("templates"))


In [5]:
# Get the target route's service patterns, their geometry, and the feeds behind them
matching_shapes, feed_dates, feed_keys = get_matching_shapes_feeds(
    engine, env, CONSTANT_MACROS, min_trips=MIN_TRIPS
)
display(feed_dates)
matching_shapes

,service_date,feed_key
0,2026-06-19,d71492afe126cd61f629b17c3c056fcd
1,2026-06-16,d71492afe126cd61f629b17c3c056fcd
2,2026-06-17,d71492afe126cd61f629b17c3c056fcd
3,2026-06-18,d71492afe126cd61f629b17c3c056fcd
5,2026-06-15,4dfefaf518766d1cfc3d254a65e4295e


,feed_key,last_seen_date,gtfs_dataset_name,shape_id,route_id,route_short_name,ct,geometry
0,4dfefaf518766d1cfc3d254a65e4295e,2026-06-15,Bay Area 511 SamTrans Schedule,ECR2068,ECR,ECR,350,"LINESTRING (-122.46879 37.70620, -122.46897 37..."
1,4dfefaf518766d1cfc3d254a65e4295e,2026-06-15,Bay Area 511 SamTrans Schedule,ECR2069,ECR,ECR,350,"LINESTRING (-122.16599 37.44395, -122.16578 37..."
2,d71492afe126cd61f629b17c3c056fcd,2026-06-19,Bay Area 511 SamTrans Schedule,ECR2069,ECR,ECR,350,"LINESTRING (-122.16599 37.44395, -122.16578 37..."
3,d71492afe126cd61f629b17c3c056fcd,2026-06-19,Bay Area 511 SamTrans Schedule,ECR2068,ECR,ECR,350,"LINESTRING (-122.46879 37.70620, -122.46897 37..."


In [6]:
# Get the schedule feed entry for every feed that was active during the window
feed_dates = get_schedule_feed_entries(engine, env, feed_dates)
feed_dates

,service_date,feed_key,base64_url,_valid_from
0,2026-06-19,d71492afe126cd61f629b17c3c056fcd,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L2RhdGFmZW...,2026-06-16 03:00:00.395507+00:00
1,2026-06-16,d71492afe126cd61f629b17c3c056fcd,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L2RhdGFmZW...,2026-06-16 03:00:00.395507+00:00
2,2026-06-17,d71492afe126cd61f629b17c3c056fcd,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L2RhdGFmZW...,2026-06-16 03:00:00.395507+00:00
3,2026-06-18,d71492afe126cd61f629b17c3c056fcd,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L2RhdGFmZW...,2026-06-16 03:00:00.395507+00:00
4,2026-06-15,4dfefaf518766d1cfc3d254a65e4295e,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L2RhdGFmZW...,2026-05-16 03:00:00.573378+00:00


In [7]:
# Calculate travel time variability
schedule_rt_stop_times = get_schedule_rt_stop_times(
    engine,
    env,
    CONSTANT_MACROS,
    feed_dates=feed_dates,
    shape_ids=matching_shapes.shape_id.to_list(),
)

output_time_difference_variance = calculate_travel_time_variability(
    schedule_rt_stop_times
)

# Calculate schedule performance
on_time = calculate_on_time_performance(
    schedule_rt_stop_times,
    early_seconds=EARLY_SECONDS,
    late_seconds=LATE_SECONDS,
)

# Calculate headway performance
schedule_rt_headways = get_schedule_rt_headways(
    engine,
    env,
    CONSTANT_MACROS,
    feed_dates=feed_dates,
    shape_ids=matching_shapes.shape_id.to_list(),
)
headway_summary = calculate_expected_wait(schedule_rt_headways)


## Results

### Travel Time Variability

#### Definition

The 80th/20th Percentile ratio of durations between specified points on a vehicle’s trip. Reported for the start of the TSP corridor to the end of the TSP corridor and between each Timepoint along the TSP corridor.

#### Reporting Grain

Per Timepoint, Service Pattern, Day Part, and Reporting Period (currently ad hoc for a week).

In [8]:
# Time variance metric per segment for northbound trip
output_time_difference_variance.loc[
    (output_time_difference_variance.feed_key == "d71492afe126cd61f629b17c3c056fcd")
    & (output_time_difference_variance.shape_id == "ECR2069")
    & (output_time_difference_variance.stop_sequence != 1)
].head(15)

,shape_id,feed_key,stop_sequence,time_period,travel_time_variability,p80_travel_time_seconds,p20_travel_time_seconds,n_rt_trips,n_service_dates
313,ECR2069,d71492afe126cd61f629b17c3c056fcd,2,Early AM,2.241379,65.0,29.0,23,3
314,ECR2069,d71492afe126cd61f629b17c3c056fcd,2,AM Peak,1.592593,129.0,81.0,23,3
315,ECR2069,d71492afe126cd61f629b17c3c056fcd,2,Midday,1.148148,93.0,81.0,43,3
316,ECR2069,d71492afe126cd61f629b17c3c056fcd,2,PM Peak,1.529412,130.0,85.0,51,3
317,ECR2069,d71492afe126cd61f629b17c3c056fcd,2,Evening,1.000000,73.0,73.0,20,3
318,ECR2069,d71492afe126cd61f629b17c3c056fcd,4,Early AM,1.703755,245.0,143.8,30,3
319,ECR2069,d71492afe126cd61f629b17c3c056fcd,4,AM Peak,1.258621,292.0,232.0,36,3
320,ECR2069,d71492afe126cd61f629b17c3c056fcd,4,Midday,1.517241,352.0,232.0,58,3
321,ECR2069,d71492afe126cd61f629b17c3c056fcd,4,PM Peak,1.451128,386.0,266.0,60,3
322,ECR2069,d71492afe126cd61f629b17c3c056fcd,4,Evening,2.216216,328.0,148.0,21,3


In [9]:
# generate map
MAP_TIME_PERIOD = "Midday"

# TODO - restrict this map to just shape id = ECR2069, feed_key = d71492afe126cd61f629b17c3c056fcd

timepoints = schedule_rt_stop_times[
    ["shape_id", "feed_key", "stop_sequence", "stop_id", "pt_geom"]
].drop_duplicates()
timepoints_gdf = gpd.GeoDataFrame(
    timepoints.drop(["pt_geom"], axis=1),
    geometry=gpd.GeoSeries(timepoints["pt_geom"].map(shapely.wkt.loads), crs=4326),
)

segments_by_shape = []
for shape_id, feed_key, shape_geom in matching_shapes[
    ["shape_id", "feed_key", "geometry"]
].itertuples(index=False):
    timepoints_for_shape = timepoints_gdf.loc[
        (timepoints_gdf.shape_id == shape_id) & (timepoints_gdf.feed_key == feed_key)
    ].copy()
    timepoints_for_shape["projected_distance"] = timepoints_for_shape.geometry.map(
        lambda point_geom: shape_geom.project(point_geom)
    )
    timepoints_for_shape["prior_projected_distance"] = timepoints_for_shape[
        "projected_distance"
    ].shift()
    timepoints_for_shape["segment_geometry"] = timepoints_for_shape[
        ["projected_distance", "prior_projected_distance"]
    ].apply(
        lambda row: (
            shapely.ops.substring(
                shape_geom, row["projected_distance"], row["prior_projected_distance"]
            )
            if not row.isna().any()
            else None
        ),
        axis=1,
    )
    segments_by_shape.append(
        timepoints_for_shape.set_geometry("segment_geometry")[
            ["shape_id", "feed_key", "stop_sequence", "stop_id", "segment_geometry"]
        ]
    )

# the leading timepoint of each pattern has no preceding stop, so no segment
segments_gdf = pd.concat(segments_by_shape).dropna(subset=["segment_geometry"])
segments_gdf = gpd.GeoDataFrame(
    segments_gdf, geometry="segment_geometry"
).set_crs(4326)

segments_for_period = segments_gdf.merge(
    output_time_difference_variance.loc[
        output_time_difference_variance["time_period"] == MAP_TIME_PERIOD
    ],
    on=["shape_id", "feed_key", "stop_sequence"],
    how="inner",
)
segments_for_period = segments_for_period.loc[
    segments_for_period.stop_sequence != 2, #TODO: there seems to be an issue where this segment overlaps the entire route, so I've manually removed it
]

# the ratio and the two percentiles behind it, so a segment can be read as more
# than a colour
for column in [
    "travel_time_variability",
    "p80_travel_time_seconds",
    "p20_travel_time_seconds",
]:
    segments_for_period[column] = segments_for_period[column].round(1)

# the shapes underneath, uncoloured. a shape carries the same geometry in every
# feed version it appears in, so draw each one once
shape_lines = matching_shapes.drop_duplicates(subset=["shape_id"])[
    ["shape_id", "geometry"]
]
travel_time_variability_map = shape_lines.explore(
    color="#888888",
    tooltip=["shape_id"],
    tiles="CartoDB positron",
    name="service patterns",
)
segments_for_period.explore(
    m=travel_time_variability_map,
    column="travel_time_variability",
    cmap="plasma",
    style_kwds={"weight": 6, "opacity": 0.9},
    tooltip=[
        "shape_id",
        "stop_sequence",
        "stop_id",
        "n_rt_trips",
        "p80_travel_time_seconds",
        "p20_travel_time_seconds",
        "travel_time_variability",
    ],
    tooltip_kwds={
        "aliases": [
            "Shape",
            "Stop sequence",
            "Stop",
            "Trips",
            "80th pct run time (s)",
            "20th pct run time (s)",
            "p80/p20 ratio",
        ]
    },
    legend_kwds={
        "caption": f"p80/p20 ratio of {MAP_TIME_PERIOD.lower()} run times into each timepoint"
    },
    name=f"{MAP_TIME_PERIOD} trip duration variability",
)
travel_time_variability_map.save(f"trip_duration_variability_{MAP_TIME_PERIOD.lower()}.html")

In [10]:
# Map of midday day part
travel_time_variability_map


### Expected Wait

#### Definition
The proportion of the time during the Day Part in which an arriving passenger would experience an Actual Headway that is greater than the Scheduled Headway.

#### Reporting Grain
Service Pattern, stop, Day Part, and Reporting Period

In [11]:
headway_summary.loc[(headway_summary.time_period == "Midday") & (headway_summary.stop_sequence != 1)]

,shape_id,stop_sequence,stop_id,time_period,n_headways,mean_rt_headway_seconds,median_rt_headway_seconds,max_rt_headway_seconds,rt_headway_variance,mean_scheduled_headway_seconds,median_scheduled_headway_seconds,scheduled_expected_wait_seconds,rt_expected_wait_seconds,excess_wait_seconds,percent_time_excess_wait
7,ECR2068,2,332170,Midday,55,1004.109091,900.0,2580.0,126133.506397,1015.018182,900.0,450.0,564.863212,114.863212,0.052149
12,ECR2068,3,332232,Midday,59,926.779661,900.0,1800.0,68173.933372,930.847458,900.0,450.0,500.169842,50.169842,0.072421
17,ECR2068,4,332238,Midday,59,925.847458,900.0,1740.0,69355.303916,930.932203,900.0,450.0,500.378767,50.378767,0.077986
22,ECR2068,5,332215,Midday,60,894.000000,900.0,1680.0,46214.237288,900.000000,900.0,450.0,472.846889,22.846889,0.076063
27,ECR2068,6,332235,Midday,60,894.000000,900.0,1740.0,50607.457627,900.000000,900.0,450.0,475.303947,25.303947,0.083893
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
757,ECR2069,71,332214,Midday,58,972.879310,990.0,2640.0,219483.897459,941.844828,900.0,450.0,599.240850,149.240850,0.229677
762,ECR2069,72,332220,Midday,57,962.947368,900.0,2640.0,243022.015038,942.947368,900.0,450.0,607.660236,157.660236,0.231745
767,ECR2069,73,332216,Midday,57,966.421053,900.0,2700.0,258635.390977,927.473684,900.0,450.0,617.021452,167.021452,0.248339
772,ECR2069,74,332171,Midday,57,968.789474,900.0,2640.0,237695.419173,927.736842,900.0,450.0,607.071245,157.071245,0.240126


In [12]:
# Map of one day part
#
# Both service patterns are drawn as plain lines for context. The metric lives on
# the stops, coloured by how much of the day part would have left a rider waiting
# longer than the schedule promised.
MAP_TIME_PERIOD = "Midday"

# The headways query carries each stop's point geometry, so the metric maps
# straight onto it - no projection onto the shape is needed the way it is for
# the segment metric in trip_duration_variability
stops = (
    schedule_rt_headways[["shape_id", "stop_sequence", "stop_id", "pt_geom"]]
    .drop_duplicates()
    .dropna(subset=["pt_geom"])
)
stops_gdf = gpd.GeoDataFrame(
    stops.drop(["pt_geom"], axis=1),
    geometry=gpd.GeoSeries(stops["pt_geom"].map(shapely.wkt.loads), crs=4326),
)

stops_for_period = stops_gdf.merge(
    headway_summary.loc[headway_summary["time_period"] == MAP_TIME_PERIOD],
    on=["shape_id", "stop_sequence", "stop_id"],
    how="inner",
)

# percent_time_excess_wait is a proportion; the map reads better in percent. The
# colour and the tooltip use the one column, so the legend matches what is hovered
stops_for_period["percent_time_excess_wait"] = (
    stops_for_period["percent_time_excess_wait"] * 100
).round(1)
for column in [
    "scheduled_expected_wait_seconds",
    "rt_expected_wait_seconds",
    "excess_wait_seconds",
]:
    stops_for_period[column] = stops_for_period[column].round(1)

# the shapes underneath, uncoloured. a shape carries the same geometry in every
# feed version it appears in, so draw each one once
shape_lines = matching_shapes.drop_duplicates(subset=["shape_id"])[
    ["shape_id", "geometry"]
]
headway_performance_map = shape_lines.explore(
    color="#888888",
    tooltip=["shape_id"],
    tiles="CartoDB positron",
    name="service patterns",
)
stops_for_period.explore(
    m=headway_performance_map,
    column="percent_time_excess_wait",
    cmap="plasma",
    marker_kwds={"radius": 6},
    style_kwds={"stroke": False, "fillOpacity": 0.9},
    tooltip=[
        "shape_id",
        "stop_sequence",
        "stop_id",
        "scheduled_expected_wait_seconds",
        "rt_expected_wait_seconds",
        "excess_wait_seconds",
        "percent_time_excess_wait",
    ],
    tooltip_kwds={
        "aliases": [
            "Shape",
            "Stop sequence",
            "Stop",
            "Scheduled expected wait (s)",
            "Actual expected wait (s)",
            "Excess wait (s)",
            "% of time waiting over schedule",
        ]
    },
    legend_kwds={
        "caption": f"% of {MAP_TIME_PERIOD.lower()} spent waiting longer than scheduled"
    },
    name=f"{MAP_TIME_PERIOD} excess wait",
)
headway_performance_map.save(f"expected_wait_{MAP_TIME_PERIOD.lower()}.html")

In [13]:
# Map of all stops for one day part

headway_performance_map

### Schedule Adherence at Timepoint

#### Definition
At each transit stop, the proportion of trips that are less than five minutes late and less than one minute early.

#### Reporting Grain
Reporting Period and Day Part. (also mapped at timepoints here)

### Average Deviation from Schedule

#### Definition

The difference between when the bus is scheduled to arrive at each stop and when the bus arrives based on real time data. Reported as 50th and 80th percentile.



In [14]:
# OTP by stop for the northbound direction midday only

on_time.by_period

,shape_id,time_period,n_timepoint_arrivals,p50_absolute_deviation_seconds,p80_absolute_deviation_seconds,proportion_early,proportion_late,proportion_not_on_time,schedule_adherence
0,ECR2068,Early AM,628,120.0,360.0,0.136943,0.259554,0.396497,0.603503
1,ECR2068,AM Peak,1188,180.0,420.0,0.182660,0.320707,0.503367,0.496633
2,ECR2068,Midday,1960,180.0,480.0,0.257143,0.262245,0.519388,0.480612
3,ECR2068,PM Peak,2520,180.0,492.0,0.326190,0.277381,0.603571,0.396429
4,ECR2068,Evening,1292,180.0,540.0,0.357585,0.260836,0.618421,0.381579
5,ECR2069,Early AM,662,120.0,360.0,0.110272,0.264350,0.374622,0.625378
6,ECR2069,AM Peak,1080,180.0,360.0,0.157407,0.295370,0.452778,0.547222
7,ECR2069,Midday,1859,240.0,480.0,0.151157,0.360947,0.512103,0.487897
8,ECR2069,PM Peak,2517,180.0,360.0,0.210171,0.244736,0.454907,0.545093
9,ECR2069,Evening,1227,180.0,360.0,0.205379,0.264874,0.470253,0.529747


In [15]:
# Map of one day part
#
# Both service patterns are drawn as plain lines for context. The metric lives on
# the stops, coloured by the share of arrivals that were not on time.
MAP_TIME_PERIOD = "Midday"

# The stop times query carries each stop's point geometry, so the metric maps
# straight onto it - no projection onto the shape is needed the way it is for
# the segment metric in trip_duration_variability
stops = (
    schedule_rt_stop_times[["shape_id", "stop_sequence", "stop_id", "pt_geom"]]
    .drop_duplicates()
    .dropna(subset=["pt_geom"])
)
stops_gdf = gpd.GeoDataFrame(
    stops.drop(["pt_geom"], axis=1),
    geometry=gpd.GeoSeries(stops["pt_geom"].map(shapely.wkt.loads), crs=4326),
)

stops_for_period = stops_gdf.merge(
    on_time.by_stop.loc[on_time.by_stop["time_period"] == MAP_TIME_PERIOD],
    on=["shape_id", "stop_sequence", "stop_id"],
    how="inner",
)

# the proportions are 0 to 1; the map reads better in percent. The colour and the
# tooltip use the one column, so the legend matches what is hovered
for column in ["proportion_early", "proportion_late", "proportion_not_on_time", "schedule_adherence"]:
    stops_for_period[column] = (stops_for_period[column] * 100).round(1)
for column in [
    "p50_absolute_deviation_seconds",
    "p80_absolute_deviation_seconds",
]:
    stops_for_period[column] = stops_for_period[column].round(1)

# the shapes underneath, uncoloured. a shape carries the same geometry in every
# feed version it appears in, so draw each one once
shape_lines = matching_shapes.drop_duplicates(subset=["shape_id"])[
    ["shape_id", "geometry"]
]
otp_map = shape_lines.explore(
    color="#888888",
    tooltip=["shape_id"],
    tiles="CartoDB positron",
    name="service patterns",
)
stops_for_period.explore(
    m=otp_map,
    column="schedule_adherence",
    cmap="plasma",
    marker_kwds={"radius": 6},
    style_kwds={"stroke": False, "fillOpacity": 0.9},
    tooltip=[
        "shape_id",
        "stop_sequence",
        "stop_id",
        "n_timepoint_arrivals",
        "p50_absolute_deviation_seconds",
        "p80_absolute_deviation_seconds",
        "proportion_early",
        "proportion_late",
        "proportion_not_on_time",
        "schedule_adherence"
    ],
    tooltip_kwds={
        "aliases": [
            "Shape",
            "Stop sequence",
            "Stop",
            "Arrivals",
            "Median deviation (s)",
            "80th pct deviation (s)",
            "% early",
            "% late",
            "% not on time",
            "On Time Performance"
        ]
    },
    legend_kwds={
        "caption": f"% of {MAP_TIME_PERIOD.lower()} arrivals on time"
    },
    name=f"{MAP_TIME_PERIOD} on time performance",
)
otp_map.save(f"on_time_performance_{MAP_TIME_PERIOD.lower()}.html")


In [16]:
otp_map